[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/finetuning-avanzado/03-evaluacion-modelos-finetuneados.ipynb)

# Evaluación de Modelos Fine-tuneados

> Aprende a evaluar la calidad de modelos fine-tuneados usando métricas automáticas
> (ROUGE, BLEU) y evaluación con LLM-as-judge. Compara modelos en un dashboard visual.

## Objetivos
- Calcular ROUGE-1, ROUGE-2 y ROUGE-L con rouge-score
- Calcular BLEU con NLTK
- Implementar LLM-as-judge con Claude para evaluación cualitativa
- Visualizar comparativa de modelos con gráfica radar

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic nltk rouge-score pandas matplotlib numpy --quiet

## 2. Dataset de evaluación con respuestas de referencia

In [ ]:
import anthropic
import nltk
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"
scorer_rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

# Dataset: pregunta + respuesta de referencia (gold standard)
DATASET_EVAL = [
    {
        "pregunta": "¿Qué es el overfitting y cómo se combate?",
        "referencia": "El overfitting ocurre cuando un modelo memoriza los datos de entrenamiento en lugar de aprender patrones generalizables. Se combate con regularización L1/L2, dropout, early stopping, más datos de entrenamiento o reduciendo la complejidad del modelo."
    },
    {
        "pregunta": "Explica qué es un transformer y su mecanismo de atención.",
        "referencia": "Un transformer es una arquitectura de red neuronal basada en el mecanismo de atención multi-cabezal, que permite procesar secuencias en paralelo. La atención calcula la relevancia entre cada par de tokens, capturando dependencias de largo alcance sin necesidad de recurrencia."
    },
    {
        "pregunta": "¿Cuál es la diferencia entre clasificación y regresión?",
        "referencia": "La clasificación predice categorías discretas (spam/no spam, perro/gato) mientras que la regresión predice valores continuos (precio de una casa, temperatura). La diferencia fundamental está en la naturaleza de la variable objetivo."
    },
    {
        "pregunta": "¿Qué son los embeddings y para qué se usan?",
        "referencia": "Los embeddings son representaciones vectoriales densas de baja dimensión que capturan el significado semántico de palabras, frases o entidades. Se usan para búsqueda semántica, sistemas de recomendación, clasificación de texto y como entrada a modelos de deep learning."
    },
]

print(f"Dataset de evaluación: {len(DATASET_EVAL)} preguntas con respuestas de referencia.")

## 3. Generar respuestas de modelos simulados

In [ ]:
def generar_respuesta_modelo(pregunta: str, calidad: str) -> str:
    """Simula respuestas de diferentes calidades de modelo con Claude."""
    instrucciones = {
        "base": "Responde de forma muy breve e incompleta, como un modelo poco entrenado.",
        "sft": "Responde de forma correcta pero un poco básica, como un modelo con instruction tuning básico.",
        "dpo": "Responde de forma precisa, clara y bien estructurada, como un modelo completamente alineado."
    }
    prompt = f"{instrucciones[calidad]}\n\nPregunta: {pregunta}"
    return client.messages.create(
        model=MODELO, max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text

print("Generando respuestas para 3 modelos simulados (base, SFT, DPO)...")
respuestas_modelos = {"base": [], "sft": [], "dpo": []}

for item in DATASET_EVAL:
    for modelo_sim in ["base", "sft", "dpo"]:
        resp = generar_respuesta_modelo(item["pregunta"], modelo_sim)
        respuestas_modelos[modelo_sim].append(resp)

print("Respuestas generadas.")

## 4. Métricas automáticas: ROUGE y BLEU

In [ ]:
smooth = SmoothingFunction().method1

def calcular_metricas(hipotesis: str, referencia: str) -> dict:
    """Calcula ROUGE y BLEU para un par hipótesis-referencia."""
    scores_rouge = scorer_rouge.score(referencia, hipotesis)
    ref_tokens = word_tokenize(referencia.lower())
    hip_tokens = word_tokenize(hipotesis.lower())
    bleu = sentence_bleu([ref_tokens], hip_tokens, smoothing_function=smooth)
    return {
        "ROUGE-1": round(scores_rouge["rouge1"].fmeasure, 4),
        "ROUGE-2": round(scores_rouge["rouge2"].fmeasure, 4),
        "ROUGE-L": round(scores_rouge["rougeL"].fmeasure, 4),
        "BLEU": round(bleu, 4)
    }

resultados_metricas = {}
for modelo_sim, respuestas in respuestas_modelos.items():
    metricas_modelo = [calcular_metricas(r, DATASET_EVAL[i]["referencia"]) for i, r in enumerate(respuestas)]
    resultados_metricas[modelo_sim] = {
        k: round(np.mean([m[k] for m in metricas_modelo]), 4) for k in ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"]
    }

df_metricas = pd.DataFrame(resultados_metricas).T
print("=== MÉTRICAS AUTOMÁTICAS (medias) ===")
print(df_metricas.to_string())

## 5. LLM-as-Judge y gráfica radar

In [ ]:
def llm_judge(pregunta: str, respuesta: str, referencia: str) -> float:
    """Claude evalúa la calidad de una respuesta respecto a la referencia (0-10)."""
    prompt = f"""Evalúa la respuesta del modelo respecto a la referencia. Puntúa de 0 a 10.

Pregunta: {pregunta}
Referencia: {referencia}
Respuesta del modelo: {respuesta}

Criterios: precisión, completitud, claridad. Responde SOLO con un número entero (0-10)."""

    resp = client.messages.create(
        model=MODELO, max_tokens=10,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()
    return float(resp.split()[0])

scores_judge = {}
for modelo_sim, respuestas in respuestas_modelos.items():
    scores = [llm_judge(DATASET_EVAL[i]["pregunta"], r, DATASET_EVAL[i]["referencia"]) / 10
              for i, r in enumerate(respuestas)]
    scores_judge[modelo_sim] = round(np.mean(scores), 4)

# Gráfica radar
categorias = ["ROUGE-1", "ROUGE-L", "BLEU", "LLM-Judge"]
N = len(categorias)
angulos = [n / float(N) * 2 * np.pi for n in range(N)] + [0]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
colores = {"base": "#e74c3c", "sft": "#f39c12", "dpo": "#2ecc71"}

for modelo_sim in ["base", "sft", "dpo"]:
    valores = [resultados_metricas[modelo_sim]["ROUGE-1"],
               resultados_metricas[modelo_sim]["ROUGE-L"],
               resultados_metricas[modelo_sim]["BLEU"],
               scores_judge[modelo_sim]] + [resultados_metricas[modelo_sim]["ROUGE-1"]]
    ax.plot(angulos, valores, "o-", linewidth=2, label=modelo_sim.upper(), color=colores[modelo_sim])
    ax.fill(angulos, valores, alpha=0.1, color=colores[modelo_sim])

ax.set_xticks(angulos[:-1])
ax.set_xticklabels(categorias, size=12)
ax.set_ylim(0, 1)
ax.set_title("Comparativa de Modelos", size=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig("evaluacion_modelos_radar.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"LLM-Judge scores: {scores_judge}")